In [0]:
gold_path = "/Volumes/data_architects/gold_layer/aggregated_data_volume/"
silver_path = "/Volumes/data_architects/silver_layer/preprocessed_data_volume/"

df_final = spark.read.format("delta") \
    .load(silver_path + "instacart_final")

df_final.display()

In [0]:
df_orders = spark.read.format("delta") \
    .load(silver_path + "orders_clean")

df_products = spark.read.format("delta") \
    .load(silver_path + "products_clean")

df_order_products = spark.read.format("delta") \
    .load(silver_path + "order_products_clean")

df_aisles = spark.read.format("delta") \
    .load(silver_path + "aisles_clean")

df_departments = spark.read.format("delta") \
    .load(silver_path + "departments_clean")

Customer Segmentation

Customer Loyalty Table

In [0]:
from pyspark.sql.functions import countDistinct, col

df_customer_orders = df_orders.groupBy("user_id") \
    .agg(countDistinct("order_id").alias("total_orders"))

df_customer_orders.orderBy(col("total_orders").desc()).display()

In [0]:
df_customer_orders.describe().display()

In [0]:
from pyspark.sql.functions import when

df_customer_orders = df_customer_orders.withColumn(
    "customer_type",
    when(col("total_orders") > 50, "High Value")
    .when(col("total_orders") > 10, "Medium")
    .otherwise("Low")
)

df_customer_orders.display()

Product combination

In [0]:
from pyspark.sql.functions import col

df_pairs = df_final.alias("a").join(
    df_final.alias("b"),
    (col("a.order_id") == col("b.order_id")) &
    (col("a.product_id") < col("b.product_id"))
)

In [0]:
df_combinations = df_pairs.groupBy(
    col("a.product_name").alias("product_1"),
    col("b.product_name").alias("product_2")
).count()

In [0]:
df_combinations.orderBy(col("count").desc()).display()

time based behaviour
Instacart can send personalized notifications based on user activity time

time grouping

In [0]:
from pyspark.sql.functions import when, col

df_time_segment = df_final.withColumn(
    "time_segment",
    when(col("order_hour_of_day") < 12, "Morning")
    .when(col("order_hour_of_day") < 18, "Afternoon")
    .otherwise("Evening/Night")
)

df_time_segment.groupBy("time_segment") \
    .count()\
    .display()

CUSTOMER CHURN RISK

Which customers are about to stop ordering 

In [0]:
from pyspark.sql.functions import avg, col

df_churn = df_orders.groupBy("user_id") \
    .agg(avg("days_since_prior_order").alias("avg_gap"))

df_churn = df_churn.withColumn(
    "risk_level",
    when(col("avg_gap") > 20, "High Risk")
    .when(col("avg_gap") > 10, "Medium Risk")
    .otherwise("Low Risk")
)

df_churn.display()


No.of items?
repeated items ?
high engagement order ?
Which orders are strong vs weak baskets?

In [0]:
from pyspark.sql.functions import count, sum, col

df_basket = df_final.groupBy("order_id") \
    .agg(
        count("product_id").alias("total_items"),
        sum("reordered").alias("repeat_items")
    )

df_basket = df_basket.withColumn(
    "basket_type",
    when(col("total_items") >= 10, "Bulk Order")
    .when(col("total_items") >= 5, "Medium Order")
    .otherwise("Small Order")
)

df_basket.display()

In [0]:
df_basket.groupBy("basket_type").count().display()